<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/XLSTM_Kfold_Gannet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries, Device Configuration, and  Model Architecture

In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

class NativesLSTMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(NativesLSTMLayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.W_x = nn.Linear(input_dim, 4 * hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)
        init_weights(self.W_x)
        init_weights(self.W_h)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            gates = self.W_x(x_t) + self.W_h(h)
            i_gate, f_gate, c_gate, o_gate = gates.chunk(4, dim=1)

            i_t = torch.exp(torch.clamp(i_gate, -5.0, 5.0))
            f_t = torch.exp(torch.clamp(f_gate, -5.0, 5.0))

            c_tilde = torch.tanh(c_gate)
            c = f_t * c + i_t * c_tilde
            o_t = torch.sigmoid(o_gate)
            h = o_t * torch.tanh(c)
            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)

class JointxLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim, num_layers=2, dropout_prob=0.2):
        super(JointxLSTMAutoencoder, self).__init__()
        self.num_layers = num_layers

        # Dynamic Encoder Stack
        self.encoder_layers = nn.ModuleList()
        current_dim = input_dim
        next_dim = hidden_dim
        for i in range(num_layers):
            self.encoder_layers.append(NativesLSTMLayer(current_dim, next_dim))
            current_dim = next_dim
            if i == 0 and num_layers > 1:
                next_dim = max(8, hidden_dim // 2)

        self.bottleneck = nn.Linear(current_dim, latent_dim)
        self.encoder_dropout = nn.Dropout(dropout_prob)

        # Dynamic Decoder Stack
        self.decoder_layers = nn.ModuleList()
        current_dim = latent_dim
        next_dim = max(8, hidden_dim // 2) if num_layers > 1 else hidden_dim
        for i in range(num_layers):
            if i == num_layers - 1:
                next_dim = hidden_dim
            self.decoder_layers.append(NativesLSTMLayer(current_dim, next_dim))
            current_dim = next_dim

        self.reconstruct = nn.Linear(hidden_dim, input_dim)
        self.decoder_dropout = nn.Dropout(dropout_prob)
        self.predictor_head = nn.Linear(latent_dim, 1)

        init_weights(self.bottleneck)
        init_weights(self.reconstruct)
        init_weights(self.predictor_head)

    def forward(self, x):
        encoded = x
        for layer in self.encoder_layers:
            encoded = layer(encoded)
            encoded = self.encoder_dropout(encoded)
        latent = self.bottleneck(encoded[:, -1, :])

        decoded_input = latent.unsqueeze(1).repeat(1, x.size(1), 1)
        decoded = decoded_input
        for layer in self.decoder_layers:
            decoded = layer(decoded)
            decoded = self.decoder_dropout(decoded)
        reconstructed_output = torch.sigmoid(self.reconstruct(decoded))
        predicted_carbon = self.predictor_head(latent)

        return reconstructed_output, predicted_carbon, latent

#Data Preprocessing and Splitting

In [9]:
FILE_PATH = '/content/rural_carbon_dataset.csv'
df = pd.read_csv(FILE_PATH)
df_processed = df.copy()

# Feature Engineering
crop_encoder = LabelEncoder()
df_processed['Crop_Type_Encoded'] = crop_encoder.fit_transform(df_processed['Crop_Type'])
df_processed['Month_sin'] = np.sin(2 * np.pi * df_processed['Month'] / 12)
df_processed['Month_cos'] = np.cos(2 * np.pi * df_processed['Month'] / 12)
df_processed['Livestock_Total'] = df_processed['Livestock_Cows'] + df_processed['Livestock_Pigs']
df_processed['Energy_per_Area'] = df_processed['Household_Energy_kWh'] / (df_processed['Crop_Area_ha'] + 1)
df_processed['Fertilizer_per_Area'] = df_processed['Fertilizer_Usage_kg'] / (df_processed['Crop_Area_ha'] + 1)

feature_cols = [
    'Month_sin', 'Month_cos', 'Crop_Type_Encoded', 'Crop_Area_ha', 'Livestock_Total',
    'Household_Energy_kWh', 'Renewable_Energy_Fraction', 'Temperature_C',
    'Rainfall_mm', 'Energy_per_Area', 'Fertilizer_per_Area'
]

X = df_processed[feature_cols].values.astype(np.float32)
y = df_processed['Carbon_Emission_tCO2'].values.astype(np.float32).reshape(-1, 1)
feat_dim = X.shape[1]


test_ratios_to_try = [0.15, 0.20, 0.25]
best_test_ratio = 0.20

best_mae = float('inf')
associated_r2 = -float('inf') # for visualization not choise

print("Evaluating Data Splitting Configurations (Targeting Lowest Scaled MAE)...")
print("-" * 80)

for ratio in test_ratios_to_try:
    X_train_val_temp, X_test_temp, y_train_val_temp, y_test_temp = train_test_split(
        X, y, test_size=ratio, random_state=42
    )

    scaler_temp_x = MinMaxScaler()
    scaler_temp_y = MinMaxScaler()

    X_tr_s = scaler_temp_x.fit_transform(X_train_val_temp)
    X_te_s = scaler_temp_x.transform(X_test_temp)

    y_tr_s = scaler_temp_y.fit_transform(y_train_val_temp)
    y_te_s = scaler_temp_y.transform(y_test_temp)

    evaluator = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    evaluator.fit(X_tr_s, y_tr_s.flatten())

    y_pred_temp = evaluator.predict(X_te_s)

    current_r2 = r2_score(y_te_s, y_pred_temp)
    current_mae = mean_absolute_error(y_te_s, y_pred_temp)

    print(f"Test Split Ratio: {ratio*100}% -> Scaled MAE: {current_mae:.6f} | Associated R²: {current_r2:.6f}")
    if current_mae < best_mae:
        best_mae = current_mae
        associated_r2 = current_r2
        best_test_ratio = ratio
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=best_test_ratio, random_state=42)

print("-" * 85)
print(f"[Decision] Optimal Split Selected: {best_test_ratio*100}% Holdout Test Set.")
print(f"[Validation Profile] Lowest Expected Scaled MAE: {best_mae:.6f} | Associated R²: {associated_r2:.6f}")
print("-" * 90)

Evaluating Data Splitting Configurations (Targeting Lowest Scaled MAE)...
--------------------------------------------------------------------------------
Test Split Ratio: 15.0% -> Scaled MAE: 0.088235 | Associated R²: 0.400582
Test Split Ratio: 20.0% -> Scaled MAE: 0.086984 | Associated R²: 0.433928
Test Split Ratio: 25.0% -> Scaled MAE: 0.087038 | Associated R²: 0.445845
-------------------------------------------------------------------------------------
[Decision] Optimal Split Selected: 20.0% Holdout Test Set.
[Validation Profile] Lowest Expected Scaled MAE: 0.086984 | Associated R²: 0.433928
------------------------------------------------------------------------------------------


#Gannet Optimization Algorithm Phase

In [10]:
# -------------------------------------------------------------------------
# 2. Gannet Optimization (Tuning via Proxy Validation Split)
# -------------------------------------------------------------------------
X_t_proxy, X_v_proxy, y_t_proxy, y_v_proxy = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)
scaler_X_p = MinMaxScaler()
scaler_y_p = MinMaxScaler()
X_t_p_scaled = np.expand_dims(scaler_X_p.fit_transform(X_t_proxy), axis=1)
X_v_p_scaled = np.expand_dims(scaler_X_p.transform(X_v_proxy), axis=1)
y_t_p_scaled = scaler_y_p.fit_transform(y_t_proxy)
y_v_p_scaled = scaler_y_p.transform(y_v_proxy)

val_inputs_X_proxy = torch.tensor(X_v_p_scaled).to(device)
val_targets_y_proxy = torch.tensor(y_v_p_scaled).to(device)

# Search Space Dimensions:
# [LR, Hidden_Size, Latent_Dim, Dropout, Weight_Decay, Batch_Size, Num_Layers, Alpha]
lb = np.array([1e-4, 16,  4, 0.0, 1e-6, 16,  1, 0.1]) # Lower bounds
ub = np.array([1e-2, 64, 16, 0.5, 1e-3, 128, 3, 2.0]) # Upper bounds

pop_size = 10
max_iter = 5

def evaluate_fitness(position):
    # Decode position values
    lr = float(position[0])
    hidden_size = int(np.round(position[1]))
    latent_dim = int(np.round(position[2]))
    dropout = float(position[3])
    weight_decay = float(position[4])
    batch_size = int(np.round(position[5]))
    num_layers = int(np.round(position[6]))
    alpha = float(position[7])

    # Fast evaluation dataloader
    t_dataset = TensorDataset(torch.tensor(X_t_p_scaled), torch.tensor(y_t_p_scaled))
    t_loader = DataLoader(t_dataset, batch_size=batch_size, shuffle=True)

    # Instantiate dynamic model
    eval_model = JointxLSTMAutoencoder(
        input_dim=feat_dim, latent_dim=latent_dim, hidden_dim=hidden_size,
        num_layers=num_layers, dropout_prob=dropout
    ).to(device)

    criterion_r = nn.MSELoss()
    criterion_p = nn.MSELoss()
    opt = optim.Adam(eval_model.parameters(), lr=lr, weight_decay=weight_decay)

    # Short optimization training loop for fitness assignment
    for epoch in range(3):
        eval_model.train()
        for bx, by in t_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            r_out, p_out, _ = eval_model(bx)
            loss = criterion_r(r_out, bx) + (alpha * criterion_p(p_out, by))
            loss.backward()
            nn.utils.clip_grad_norm_(eval_model.parameters(), 5.0)
            opt.step()

    # Evaluation on Validation set
    eval_model.eval()
    with torch.no_grad():
        v_r, v_p, _ = eval_model(val_inputs_X_proxy)
        val_loss = criterion_r(v_r, val_inputs_X_proxy).item() + (alpha * criterion_p(v_p, val_targets_y_proxy).item())
    return val_loss

# Initialize Gannet Population
gannet_positions = np.random.uniform(lb, ub, (pop_size, len(lb)))
fitness_scores = np.array([evaluate_fitness(p) for p in gannet_positions])

best_idx = np.argmin(fitness_scores)
best_gannet_score = fitness_scores[best_idx]
best_gannet_position = gannet_positions[best_idx].copy()

print("Executing Gannet Optimization Strategy...")
for iteration in range(max_iter):
    for i in range(pop_size):
        # Gannet Exploration and Exploitation mathematical updating mechanics
        t = 1 - (iteration / max_iter)
        c = 0.2 * (t ** 2)
        v = np.random.randn(*lb.shape)

        if np.random.rand() < 0.5:
            # Dive exploration phase
            gannet_positions[i] = gannet_positions[i] + c * v * (gannet_positions[i] - best_gannet_position)
        else:
            # Trajectory exploitation phase
            gannet_positions[i] = best_gannet_position + c * np.random.rand() * (best_gannet_position - gannet_positions[i])

        # Bound enforcement
        gannet_positions[i] = np.clip(gannet_positions[i], lb, ub)

        # Re-evaluate
        fit = evaluate_fitness(gannet_positions[i])
        if fit < fitness_scores[i]:
            fitness_scores[i] = fit
            if fit < best_gannet_score:
                best_gannet_score = fit
                best_gannet_position = gannet_positions[i].copy()

    print(f"Gannet Iteration [{iteration+1}/{max_iter}] -> Best Discovered Fitness: {best_gannet_score:.6f}")

# Extract Optimized Global Best Parameters
best_lr = float(best_gannet_position[0])
best_hidden_size = int(np.round(best_gannet_position[1]))
best_latent_dim = int(np.round(best_gannet_position[2]))
best_dropout = float(best_gannet_position[3])
best_weight_decay = float(best_gannet_position[4])
best_batch_size = int(np.round(best_gannet_position[5]))
best_num_layers = int(np.round(best_gannet_position[6]))
best_alpha = float(best_gannet_position[7])

print("\nGannet Search Completed. Optimal Hyperparameters Parsed.")

Executing Gannet Optimization Strategy...
Gannet Iteration [1/5] -> Best Discovered Fitness: 0.031426
Gannet Iteration [2/5] -> Best Discovered Fitness: 0.031426
Gannet Iteration [3/5] -> Best Discovered Fitness: 0.031426
Gannet Iteration [4/5] -> Best Discovered Fitness: 0.031426
Gannet Iteration [5/5] -> Best Discovered Fitness: 0.031280

Gannet Search Completed. Optimal Hyperparameters Parsed.


#K-Fold Cross Validation/Ensembling & Model Training


In [11]:
# -------------------------------------------------------------------------
# K-Fold Cross Validation & Model Training
# -------------------------------------------------------------------------
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

epochs = 100
patience = 20
models_ensemble = []
scalers_ensemble = []  # Store local fold scalers to prevent inference mismatch

criterion_recon = nn.MSELoss()
criterion_pred = nn.MSELoss()

print(f"\nStarting {n_splits}-Fold Cross-Validation Protocol...")
print("-" * 90)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val, y_train_val)):
    print(f"\n--- Training Fold {fold + 1}/{n_splits} ---")

    # Isolate fold-specific splits
    X_tr, X_va = X_train_val[train_idx], X_train_val[val_idx]
    y_tr, y_va = y_train_val[train_idx], y_train_val[val_idx]

    # Fit scalers locally on fold training data
    scaler_X_fold = MinMaxScaler()
    scaler_y_fold = MinMaxScaler()

    X_tr_s = np.expand_dims(scaler_X_fold.fit_transform(X_tr), axis=1)
    X_va_s = np.expand_dims(scaler_X_fold.transform(X_va), axis=1)
    y_tr_s = scaler_y_fold.fit_transform(y_tr)
    y_va_s = scaler_y_fold.transform(y_va)

    scalers_ensemble.append((scaler_X_fold, scaler_y_fold))

    train_dataset = TensorDataset(torch.tensor(X_tr_s), torch.tensor(y_tr_s))
    train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)

    val_inputs_X = torch.tensor(X_va_s).to(device)
    val_targets_y = torch.tensor(y_va_s).to(device)

    # Initialize separate instance per fold
    fold_model = JointxLSTMAutoencoder(
        input_dim=feat_dim, latent_dim=best_latent_dim, hidden_dim=best_hidden_size,
        num_layers=best_num_layers, dropout_prob=best_dropout
    ).to(device)

    optimizer = optim.Adam(fold_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    patience_counter = 0
    checkpoint_path = f'best_model_fold_{fold}.pth'

    for epoch in range(epochs):
        fold_model.train()
        train_total_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            recon_out, pred_carbon, _ = fold_model(batch_x)

            loss_recon = criterion_recon(recon_out, batch_x)
            loss_pred = criterion_pred(pred_carbon, batch_y)
            loss_total = loss_recon + (best_alpha * loss_pred)

            loss_total.backward()
            nn.utils.clip_grad_norm_(fold_model.parameters(), max_norm=5.0)
            optimizer.step()

            train_total_loss += loss_total.item() * batch_x.size(0)
        train_total_loss /= len(train_loader.dataset)

        # Validation checks
        fold_model.eval()
        with torch.no_grad():
            val_recon, val_pred, _ = fold_model(val_inputs_X)
            val_recon_loss = criterion_recon(val_recon, val_inputs_X).item()
            val_pred_loss = criterion_pred(val_pred, val_targets_y).item()
            val_total_loss = val_recon_loss + (best_alpha * val_pred_loss)

        scheduler.step(val_total_loss)

        if val_total_loss < best_val_loss:
            best_val_loss = val_total_loss
            patience_counter = 0
            safe_config_tensor = torch.tensor(best_gannet_position, dtype=torch.float32)
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': fold_model.state_dict(),
                'best_val_loss': float(best_val_loss),
                'config_tensor': safe_config_tensor
            }, checkpoint_path)
        else:
            patience_counter += 1

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:03d}/{epochs}] -> Train Loss: {train_total_loss:.6f} | Total Val Loss: {val_total_loss:.6f}")

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1} for Fold {fold+1}.")
            break

    # Load best state for this fold and append to active ensemble array
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=True)
        fold_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Restored optimized state for Fold {fold+1} from epoch {checkpoint['epoch']}")

    fold_model.eval()
    models_ensemble.append(fold_model)

print("-" * 90)
print("Optimized Multi-Fold Model Training Phase Finished.")


Starting 5-Fold Cross-Validation Protocol...
------------------------------------------------------------------------------------------

--- Training Fold 1/5 ---
Epoch [001/100] -> Train Loss: 0.129783 | Total Val Loss: 0.069231
Epoch [020/100] -> Train Loss: 0.032529 | Total Val Loss: 0.027788
Epoch [040/100] -> Train Loss: 0.030622 | Total Val Loss: 0.026796
Epoch [060/100] -> Train Loss: 0.030146 | Total Val Loss: 0.026575
Epoch [080/100] -> Train Loss: 0.029917 | Total Val Loss: 0.026512
Early stopping triggered at epoch 91 for Fold 1.
Restored optimized state for Fold 1 from epoch 71

--- Training Fold 2/5 ---
Epoch [001/100] -> Train Loss: 0.120979 | Total Val Loss: 0.064300
Epoch [020/100] -> Train Loss: 0.033307 | Total Val Loss: 0.028938
Epoch [040/100] -> Train Loss: 0.030380 | Total Val Loss: 0.026714
Epoch [060/100] -> Train Loss: 0.030198 | Total Val Loss: 0.026533
Early stopping triggered at epoch 76 for Fold 2.
Restored optimized state for Fold 2 from epoch 56

--- Tra

#Full Performance Evaluation Metrics

In [15]:
def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Containers for gathering individual fold predictions
ensemble_recon_preds = []
ensemble_carbon_preds = []
ensemble_scaled_carbon_trues = []
ensemble_scaled_carbon_preds = []
latent_shapes = []
# Perform predictions through each trained fold using its corresponding scaler mapping
for fold_idx, (fold_model, (scaler_X_f, scaler_y_f)) in enumerate(zip(models_ensemble, scalers_ensemble)):

    # Standardize out-of-sample data matching fold distribution parameters
    X_test_scaled_f = np.expand_dims(scaler_X_f.transform(X_test), axis=1)
    y_test_scaled_f = scaler_y_f.transform(y_test)

    test_inputs_X_tensor = torch.tensor(X_test_scaled_f).to(device)

    with torch.no_grad():
        final_recon_f, final_pred_f, final_latent_f = fold_model(test_inputs_X_tensor)
        if fold_idx == 0:
            latent_shapes = final_latent_f.shape

    # Inverse transform prediction tensors back to original target scales
    pred_carbon_scaled = final_pred_f.cpu().numpy()
    ensemble_scaled_carbon_preds.append(pred_carbon_scaled.flatten())
    ensemble_scaled_carbon_trues.append(y_test_scaled_f.flatten())
    # Flatten features out for standard 2D operations
    pred_carbon_unscaled = scaler_y_f.inverse_transform(pred_carbon_scaled)
    X_recon_flat_f = final_recon_f.cpu().numpy().reshape(-1, feat_dim)
    X_recon_unscaled = scaler_X_f.inverse_transform(X_recon_flat_f)

    ensemble_carbon_preds.append(pred_carbon_unscaled)
    ensemble_recon_preds.append(X_recon_unscaled)

# Ensemble Average across all folds
final_y_pred_scaled = np.mean(ensemble_scaled_carbon_preds, axis=0)
final_y_true_scaled = np.mean(ensemble_scaled_carbon_trues, axis=0)

final_y_pred_np = np.mean(ensemble_carbon_preds, axis=0).flatten()
final_y_true_np = y_test.flatten()
final_X_recon_np = np.mean(ensemble_recon_preds, axis=0)
X_true_flat = X_test

# Compute performance evaluations
recon_r2 = r2_score(X_true_flat, final_X_recon_np)
pred_r2 = r2_score(final_y_true_np, final_y_pred_np)
pred_rmse = np.sqrt(mean_squared_error(final_y_true_np, final_y_pred_np))
pred_mape = calculate_mape(final_y_true_np, final_y_pred_np)

pred_mae_unscaled = mean_absolute_error(final_y_true_np, final_y_pred_np)
pred_mae_scaled = mean_absolute_error(final_y_true_scaled, final_y_pred_scaled)

print("\n==================== Final Gannet-CV ENSEMBLE TEST Metrics ====================")
print(f"Evaluated Architecture Layout: {best_num_layers} xLSTM Layers | Alpha Weight: {best_alpha:.4f}")
print(f"Selected Execution Controls  : LR: {best_lr:.5f} | Batch Size: {best_batch_size} | Dropout: {best_dropout:.2f}")
print("-" * 88)
print(f"Reconstruction Task Test R² (X Integrity)     = {recon_r2:.6f}")
print(f"Unscaled Prediction Task Test R² (Carbon Y)   = {pred_r2:.6f}")
print(f"Unscaled Prediction Task Test RMSE           = {pred_rmse:.6f}")
print(f"Unscaled Prediction Task Test MAPE           = {pred_mape:.2f}%")
print("-" * 88)
print(f" Scaled MAE (Normalized CO2)       = {pred_mae_scaled:.6f}  ")
print(f" Unscaled MAE (Original tCO2)      = {pred_mae_unscaled:.6f} ton co2")
print("========================================================================================")


==================== Final Gannet-CV ENSEMBLE TEST Metrics ====================
Evaluated Architecture Layout: 1 xLSTM Layers | Alpha Weight: 2.0000
Selected Execution Controls  : LR: 0.00527 | Batch Size: 20 | Dropout: 0.26
----------------------------------------------------------------------------------------
Reconstruction Task Test R² (X Integrity)     = 0.976925
Unscaled Prediction Task Test R² (Carbon Y)   = 0.469859
Unscaled Prediction Task Test RMSE           = 4.919552
Unscaled Prediction Task Test MAPE           = 77.03%
----------------------------------------------------------------------------------------
 Scaled MAE (Normalized CO2)       = 0.084604  
 Unscaled MAE (Original tCO2)      = 3.869635 ton co2
